In [1]:
"""
MAP - Charting Student Math Misunderstandings - Inference v8.1 (Single-Turn RAG)
架構升級：
1. 修正 RAG 崩塌問題：將檢索到的歷史案例「內嵌」於單輪 user prompt 中，嚴格契合 LoRA 結構。
2. 沿用輕量 BM25 檢索器，不拖慢速度。
3. 沿用 GPU 原地運算技術，拒絕巨大 Logits 搬移，死守 9 小時限時。
"""

import os
import re
import time
import math
import pandas as pd
import numpy as np
import torch
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from tqdm import tqdm

# ==== 路徑 ====
BASE_MODEL_PATH = "/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1"
ADAPTER_PATH = "/kaggle/input/datasets/alextsai2004/gemma-math-misunderstanding-lora/best_gemma_lora_model"
TEST_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/test.csv"
TRAIN_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/train.csv"
SAMPLE_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/sample_submission.csv"
OUTPUT_CSV = "/kaggle/working/submission.csv"

# ==== 超參 (回歸 V7 穩定高分設定) ====
BATCH_SIZE = 16           
SUB_BATCH_SIZE = 32       
NUM_BEAMS = 5             
NUM_RETURN = 5            
MAX_NEW_TOKENS = 24       

# ==== Sample submission 骨架 ====
sample = pd.read_csv(SAMPLE_CSV)
ROW_ID_COL = sample.columns[0]
PRED_COL = sample.columns[1]

# ==== BM25 檢索器 ====
def tokenize_text(text):
    if not isinstance(text, str):
        return []
    return re.findall(r'\w+', text.lower())

class LightweightBM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.doc_lens = [len(d) for d in docs]
        self.avg_doc_len = sum(self.doc_lens) / len(docs) if docs else 1
        self.N = len(docs)
        self.index = {}
        df = {}
        for doc_id, doc in enumerate(docs):
            counts = Counter(doc)
            for term, tf in counts.items():
                if term not in self.index:
                    self.index[term] = []
                self.index[term].append((doc_id, tf))
                df[term] = df.get(term, 0) + 1
        self.idf = {}
        for term, f in df.items():
            self.idf[term] = math.log(1 + (self.N - f + 0.5) / (f + 0.5))
            
    def retrieve_top_1(self, query_tokens):
        scores = {}
        for term in query_tokens:
            if term not in self.index:
                continue
            idf = self.idf[term]
            for doc_id, tf in self.index[term]:
                num = idf * tf * (self.k1 + 1)
                den = tf + self.k1 * (1 - self.b + self.b * self.doc_lens[doc_id] / self.avg_doc_len)
                scores[doc_id] = scores.get(doc_id, 0.0) + (num / den)
        if not scores:
            return 0  
        return max(scores, key=scores.get)

# ==== 讀取訓練集與建立索引 ====
print("Loading train set & building BM25 index...")
train_df = pd.read_csv(TRAIN_CSV)
for col in ["QuestionText", "MC_Answer", "StudentExplanation"]:
    train_df[col] = train_df[col].fillna("")

train_df["target"] = (
    train_df["Category"].astype(str) + ":" +
    train_df["Misconception"].fillna("NA").astype(str)
)
unique_labels = sorted(train_df["target"].unique().tolist())
unique_labels_set = set(unique_labels)
fallback_labels = train_df["target"].value_counts().head(3).index.tolist()
if not fallback_labels:
    fallback_labels = ["NA:NA", "NA:NA", "NA:NA"]

train_corpus = [tokenize_text(text) for text in train_df["StudentExplanation"]]
bm25_detector = LightweightBM25(train_corpus)

# ==== 核心改動：單輪內嵌 RAG Prompt 模板 ====
def build_single_turn_rag_prompt(row):
    # 1. 現場檢索相似案例
    q_tokens = tokenize_text(row["StudentExplanation"])
    matched_idx = bm25_detector.retrieve_top_1(q_tokens)
    matched_row = train_df.iloc[matched_idx]
    
    # 2. 將參考案例包裝成內部區塊 (不切換對話回合)
    reference_block = (
        "### Reference Case (Similar Historical Example):\n"
        f"Reference Question: {matched_row['QuestionText']}\n"
        f"Reference Correct Answer: {matched_row['MC_Answer']}\n"
        f"Reference Student Explanation: {matched_row['StudentExplanation']}\n"
        f"Reference Predicted Label: {matched_row['target']}\n\n"
    )
    
    # 3. 完美的單輪對話結構
    user_content = (
        "You are a math misconception classifier.\n"
        "Given the question, the correct answer, and the student's explanation, "
        "predict the final label in the format `Category:Misconception`.\n"
        "If the category is not a misconception type, use `NA` for the misconception part.\n\n"
        f"{reference_block}"
        "### Target Task:\n"
        f"Question: {row['QuestionText']}\n"
        f"Correct answer: {row['MC_Answer']}\n"
        f"Student explanation: {row['StudentExplanation']}\n\n"
        "Return only the label."
    )
    return f"<start_of_turn>user\n{user_content}<end_of_turn>\n<start_of_turn>model\n"

# ==== Model 載入 ====
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
tokenizer.padding_side = "left"  

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH, torch_dtype=torch.bfloat16, device_map="auto"
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model = model.merge_and_unload()
torch.cuda.empty_cache()  
model.eval()
model.config.use_cache = True  
device = next(model.parameters()).device

pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
tokenizer.pad_token_id = pad_id
end_of_turn_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
stop_ids = [tokenizer.eos_token_id]
if end_of_turn_id and end_of_turn_id != tokenizer.unk_token_id:
    stop_ids.append(end_of_turn_id)

# ==== 測試集載入 ====
test_df = pd.read_csv(TEST_CSV).reset_index(drop=True)
for col in ["QuestionText", "MC_Answer", "StudentExplanation"]:
    test_df[col] = test_df[col].fillna("")

def clean_label(text):
    if not text: return ""
    label = text.splitlines()[0].strip()
    if " " in label: label = label.split(" ")[0]
    return label

@torch.no_grad()
def beam_generate_batch(prompts):
    tokenizer.padding_side = "left"
    enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1200).to(device)
    outputs = model.generate(
        **enc, max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS, num_return_sequences=NUM_RETURN,
        do_sample=False, early_stopping=True, eos_token_id=stop_ids, pad_token_id=pad_id, use_cache=True,
    )
    prompt_len = enc["input_ids"].shape[1]
    outputs = outputs.view(len(prompts), NUM_RETURN, -1)
    results = []
    for i in range(len(prompts)):
        valid, seen = [], set()
        for k in range(NUM_RETURN):
            gen = outputs[i, k, prompt_len:]
            text = tokenizer.decode(gen, skip_special_tokens=True).strip()
            label = clean_label(text)
            if label in unique_labels_set and label not in seen:
                valid.append(label)
                seen.add(label)
        results.append(valid)
    return results

@torch.no_grad()
def score_candidates_batched(prompts, candidates_per_prompt):
    flat_sequences, flat_meta = [], []
    for pi, (prompt, cands) in enumerate(zip(prompts, candidates_per_prompt)):
        if not cands: continue
        prompt_ids = tokenizer.encode(prompt, add_special_tokens=True)
        for ci, cand in enumerate(cands):
            cand_ids = tokenizer.encode(cand, add_special_tokens=False)
            cand_ids.append(end_of_turn_id if end_of_turn_id else tokenizer.eos_token_id)
            flat_sequences.append(prompt_ids + cand_ids)
            flat_meta.append((pi, ci, len(cand_ids)))

    if not flat_sequences: return [[] for _ in prompts]
    B, max_len = len(flat_sequences), max(len(s) for s in flat_sequences)
    input_ids = torch.full((B, max_len), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((B, max_len), dtype=torch.long)
    label_starts = []
    for j, seq in enumerate(flat_sequences):
        input_ids[j, :len(seq)] = torch.tensor(seq, dtype=torch.long)
        attention_mask[j, :len(seq)] = 1
        label_starts.append(len(seq) - flat_meta[j][2])

    scores_per_prompt = [[0.0] * len(c) for c in candidates_per_prompt]
    for m in range(0, B, SUB_BATCH_SIZE):
        sub_input_ids = input_ids[m:m+SUB_BATCH_SIZE].to(device)
        sub_attention_mask = attention_mask[m:m+SUB_BATCH_SIZE].to(device)
        sub_logits = model(input_ids=sub_input_ids, attention_mask=sub_attention_mask).logits
        
        for idx, (pi, ci, L) in enumerate(flat_meta[m:m+SUB_BATCH_SIZE]):
            ls = label_starts[m+idx]
            slice_logits = sub_logits[idx, ls - 1:ls - 1 + L, :].float()
            log_probs = torch.log_softmax(slice_logits, dim=-1)
            target = torch.tensor(flat_sequences[m+idx][-L:], device=device)
            tok_lp = log_probs.gather(1, target.unsqueeze(1)).squeeze(1)
            scores_per_prompt[pi][ci] = tok_lp.mean().item()
            
    return scores_per_prompt

# ==== 主迴圈 ====
print("\nStart Safe Single-Turn RAG Inference...")
pred_dict = {}

try:
    for start in tqdm(range(0, len(test_df), BATCH_SIZE), desc="Batch"):
        batch_df = test_df.iloc[start:start + BATCH_SIZE]
        prompts = [build_single_turn_rag_prompt(r) for _, r in batch_df.iterrows()]

        candidates = beam_generate_batch(prompts)
        scores = score_candidates_batched(prompts, candidates)

        for i, (_, row) in enumerate(batch_df.iterrows()):
            cands = candidates[i]
            if not cands:
                top3 = list(fallback_labels[:3])
            else:
                ranked = sorted(zip(cands, scores[i]), key=lambda x: -x[1])
                top3 = [c for c, _ in ranked]
                for fb in fallback_labels:
                    if len(top3) >= 3: break
                    if fb not in top3: top3.append(fb)
            while len(top3) < 3: top3.append(fallback_labels[0])
            pred_dict[row["row_id"]] = " ".join(top3[:3])

except Exception as e:
    print(f"\n[CRITICAL ERROR] {str(e)}")
    for _, row in test_df.iterrows():
        if row["row_id"] not in pred_dict: pred_dict[row["row_id"]] = " ".join(fallback_labels[:3])

submission = sample.copy()
submission[PRED_COL] = submission[ROW_ID_COL].map(pred_dict)
submission[PRED_COL] = submission[PRED_COL].fillna(" ".join(fallback_labels[:3]))
submission.to_csv(OUTPUT_CSV, index=False)
print(f"\n[SUCCESS] Saved optimized RAG submission to {OUTPUT_CSV}")

Loading train set & building BM25 index...
Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(



Start Safe Single-Turn RAG Inference...


Batch: 100%|██████████| 1/1 [00:08<00:00,  8.03s/it]


[SUCCESS] Saved optimized RAG submission to /kaggle/working/submission.csv
